# Exercise 2.5: Loading, Inspecting and Cleaning (Angola IEA)

`IEA_2025_IV_TRIM_IND.sav`: the Inquerito ao Emprego em Angola, 4th quarter 2025,
published by INE Angola. 53,353 people, 206 columns, labels in Portuguese.

You will practice: reading the codebook to choose columns, loading with value
labels, recasting what the labels got wrong, and separating missing by design
from missing by error.

**PT:** `IEA_2025_IV_TRIM_IND.sav`: Inquerito ao Emprego em Angola, IV trimestre
2025, publicado pelo INE Angola. 53.353 pessoas, 206 colunas, etiquetas em
portugues.

Vai praticar: ler o dicionario de variaveis para escolher colunas, carregar com
etiquetas de valores, corrigir tipos que as etiquetas estragaram, e distinguir
valores em falta por desenho dos valores em falta por erro.

> **Pipeline:** reads `0_raw/`, writes `10_cleaned/`.

### Path Setup (run first)

Define the country folder once, then join sub folder and file name onto it.

**PT:** Defina a pasta do pais uma vez e depois junte a subpasta e o nome do
ficheiro.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'

EMPLOYMENT_DIR = 'employment_survey'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, EMPLOYMENT_DIR, RAW_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1: Read the codebook before choosing columns

206 columns named `ATW_PAY` or `SRH_AVN` tell you nothing. The file carries a
description for every one of them, so read that first and let it drive the
selection.

`pd.read_spss` does not expose these descriptions, so this one cell uses
`pyreadstat` with `metadataonly=True`, which reads the header without loading
any data.

**PT:** 206 colunas com nomes como `ATW_PAY` nao dizem nada. O ficheiro traz uma
descricao de cada uma, entao leia isso primeiro e deixe a descricao guiar a
escolha. `pd.read_spss` nao da acesso a estas descricoes, por isso esta celula
usa `pyreadstat` com `metadataonly=True`, que le o cabecalho sem carregar dados.

In [ ]:
import pyreadstat

_, meta = pyreadstat.read_sav(  # your code here: metadataonly=True )
codebook =   # your code here: meta.column_names_to_labels

print('Variables described in the file:', len(codebook))
for name in ['PROV', 'DEM_AGE', 'ATW_PAY', 'SRH_AVN']:
    print(f'{name:12s} {codebook[name]}')

---

## Task 2: Find columns by what they measure

Search the descriptions instead of guessing at names. The number of matches is
itself informative: a precise concept returns one column, a vague one returns
twenty and forces you to choose.

**PT:** Pesquise nas descricoes em vez de adivinhar nomes. O numero de
resultados ja e informativo: um conceito preciso devolve uma coluna, um conceito
vago devolve vinte e obriga a escolher.

In [ ]:
def find_columns(codebook, keyword):
    """Columns whose description contains `keyword`, case insensitive."""
    # your code here: keep the pairs whose label contains keyword,
    # case insensitive, skipping columns with no label
    return


for keyword in ['sexo', 'idade', 'provincia', 'horas']:
    hits = find_columns(codebook, keyword)
    print(f'{keyword:12s} {len(hits):3d} matches')

In [ ]:
# A vague keyword needs reading, not trusting
for name, label in find_columns(codebook, 'idade').items():
    print(f'{name:16s} {label[:70]}')

**Questions:**

- How many columns match `sexo`? And `idade`? Why the difference?
- Read the `idade` matches. Which one is the respondent's own age, and what are
  the others?
- Is searching the codebook enough on its own?

**PT:** Quantas colunas correspondem a `sexo`? E a `idade`? Porque a diferenca?
Qual delas e a idade do proprio inquirido? Chega pesquisar o dicionario?

---

## Task 3: Select the columns and keep their descriptions

Keep a `descriptions` dictionary alongside the data. Six months from now it is
the only thing that will tell you what `SRH_DES` meant.

**PT:** Guarde um dicionario `descriptions` junto com os dados. Daqui a seis
meses e a unica coisa que lhe dira o que significava `SRH_DES`.

In [ ]:
SPSS_COLS = [
    'NIDF', 'PPNO', 'G_06_ID_IEA', 'PROV', 'AREA_RESID', 'G_15_TRIMESTRE',
    'DEM_REL', 'DEM_SEX', 'DEM_AGE', 'DEM_MRT', 'DEM_EDL', 'S03_01',
    'ATW_PAY', 'ATW_PFT', 'ATW_FAM', 'ABS_JOB',
    'SRH_JOB', 'SRH_BUS', 'SRH_AVN', 'SRH_AVL', 'SRH_DES',
    'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'MJJ_EMP_REL', 'GHVEDT',
    'POND_IEA_IV_TRIM_2025_IND',
]

descriptions =   # your code here: lower cased name -> description, for SPSS_COLS

print('Selected:', len(SPSS_COLS), 'of', len(codebook))
pd.Series(descriptions).str.slice(0, 80).to_frame('description')

**Questions:**

- How many of the 206 columns did you keep?
- Why lower case the dictionary keys?
- What goes wrong if you analyse a variable without reading its description?

**PT:** Quantas das 206 colunas manteve? Porque por as chaves em minusculas? O
que corre mal se analisar uma variavel sem ler a descricao?

---

## Task 4: Load with value labels

Keep `convert_categoricals=True`, the default, so coded variables arrive as the
Portuguese labels stored in the file. Lower case the names: they are already
underscore separated, so that is enough to make them snake case, and nothing is
renamed, so every name still matches the codebook.

**PT:** Mantenha `convert_categoricals=True` para que as variaveis codificadas
cheguem com as etiquetas em portugues. Ponha os nomes em minusculas: nada e
renomeado, por isso cada nome continua a corresponder ao dicionario.

In [ ]:
df = pd.read_spss(  # your code here: usecols and convert_categoricals=True )
df.columns =   # your code here: lower case the names

print('Loaded:', df.shape)
df.head()

In [ ]:
df.tail()

**Questions:**

- How many rows and columns did you load?
- Look at `prov` and `dem_sex`. Text or numbers? What decided that?
- What do you gain by keeping the questionnaire's own names?

**PT:** Quantas linhas e colunas carregou? `prov` e `dem_sex` sao texto ou
numeros? O que ganha ao manter os nomes originais?

---

## Task 5: Summary statistics

Look hard at every `min` and `max`, and at any `count` below 53,353.

**PT:** Olhe com atencao para cada `min` e `max`, e para qualquer `count`
abaixo de 53.353.

In [ ]:
df.describe(  # your code here: include='all' ).T

**Questions:**

- Look at the `max` of `wkt_ushrstot`. Is that a possible working week?
- `dem_age` runs 0 to 120. Which end is real?
- What is `ghvedt` really?
- Which columns are answered by only a fifth of the sample, and why?

**PT:** O `max` de `wkt_ushrstot` e uma semana de trabalho possivel? Que extremo
de `dem_age` e real? O que e `ghvedt`? Que colunas so um quinto responde?

---

## Task 6: Explore categories

`value_counts()` shows what is actually in a coded column. Always pass
`dropna=False`.

**PT:** `value_counts()` mostra o que esta realmente numa coluna codificada.
Use sempre `dropna=False`.

In [ ]:
print(df['prov'].  # your code here: value_counts(dropna=False).sort_index() )

In [ ]:
print(df['area_resid'].value_counts(dropna=False))
print()
print(df['dem_sex'].value_counts(dropna=False))
print()
print('Distinct households:', df['nidf'].nunique())
print(df['nidf'].value_counts().describe())

**Questions:**

- How many provinces appear, and which is largest?
- How many households, and how many people each on average?

**PT:** Quantas provincias aparecem e qual e a maior? Quantos agregados, e
quantas pessoas em media?

---

## Task 7: Read the dtypes critically

The dtypes decide the next hour of work.

**PT:** Os tipos de dados determinam a proxima hora de trabalho.

In [ ]:
print(type(df))
print(type(df['dem_age']))

In [ ]:
df.  # your code here

**Questions:**

- Which columns are `category` and which `float64`? What decided that?
- Three dtypes are wrong for what the column means. Find them.

**PT:** Que colunas sao `category` e quais `float64`? Tres tipos estao errados
para o que a coluna significa. Encontre-os.

---

## Task 8: Recast what the file got wrong

`mjt_syr` is the year somebody started their main job. The value `9997` carries
the label `NAO SABE`, so pandas concluded the whole column is categorical, and a
year you cannot subtract is useless. `pd.to_numeric` with `errors='coerce'`
fixes it in one move: real years convert, the text label becomes `NaN`.

The hours columns carry the same `997` sentinel but no label for it, so they
stayed numeric and the sentinel has to be replaced by hand.

**PT:** `mjt_syr` e o ano em que a pessoa comecou o emprego principal. O valor
`9997` tem a etiqueta `NAO SABE`, por isso o pandas tornou a coluna categorica, e
um ano que nao se pode subtrair nao serve. `pd.to_numeric` com `errors='coerce'`
resolve: os anos reais convertem, a etiqueta vira `NaN`. As colunas de horas tem
o sentinela `997` sem etiqueta, por isso continuam numericas e o sentinela tem de
ser substituido a mao.

In [ ]:
print('mjt_syr dtype before:', df['mjt_syr'].dtype)
print('non numeric categories:',
      [c for c in df['mjt_syr'].cat.categories if not isinstance(c, float)])

df['mjt_syr'] = pd.to_numeric(  # your code here: astype('object'), errors='coerce' )

print()
print('mjt_syr dtype after: ', df['mjt_syr'].dtype)
print('range:', df['mjt_syr'].min(), 'to', df['mjt_syr'].max())
print('mean: ', round(df['mjt_syr'].mean(), 1))

In [ ]:
print('hours max before:', df['wkt_ushrstot'].max())

df['wkt_ushrstot'] = df['wkt_ushrstot'].  # your code here: replace sentinels with np.nan
df['wkt_achrstot'] = df['wkt_achrstot'].  # your code here: replace sentinels with np.nan

print('hours max after: ', df['wkt_ushrstot'].max())

In [ ]:
# Identifiers are labels, not quantities. Float to integer to string, or the
# trailing .0 survives and joins to nothing.
print('Before:', df['nidf'].head(3).tolist())

for col in ['nidf', 'ppno', 'g_06_id_iea']:
    df[col] =   # your code here: int64 then string

print('After: ', df['nidf'].head(3).tolist())

In [ ]:
# ghvedt is the float 20251204.0. Int64 tolerates the missing values.
df['ghvedt'] = pd.to_datetime(  # your code here: Int64 then string, format='%Y%m%d' )

print('dtype:', df['ghvedt'].dtype)
print('Range:', df['ghvedt'].min(), 'to', df['ghvedt'].max())
print('Missing (NaT):', df['ghvedt'].isna().sum())
print()
print(df['ghvedt'].dt.month.value_counts(dropna=False).sort_index())

**Questions:**

- What dtype does `mjt_syr` end up with, and what is its mean? What happened to
  the `NAO SABE` entries?
- Why did the hours columns need a manual `replace` when `mjt_syr` did not?
- What does the identifier look like if you skip the `int64` step?
- Look at the date range and month counts. Is this really the 4th quarter? What
  is missing entirely?

**PT:** Que tipo tem `mjt_syr` no fim e qual e a media? Porque as colunas de
horas precisaram de `replace` manual? Isto e mesmo o IV trimestre? O que falta?

---

## Task 9: Subset to inspect

Filtering here is for looking, not fixing.

**PT:** Aqui filtramos para observar, nao para corrigir.

In [ ]:
print('wkt_ushrstot > 100:', (df['wkt_ushrstot'] > 100).sum())
df[df['wkt_ushrstot'] > 100][['nidf', 'wkt_ushrstot', 'wkt_achrstot']].head()

In [ ]:
# Each condition needs its own parentheses
old_and_working = df[(df['dem_age'] >= 65) & (df['wkt_ushrstot'] > 40)]
print('People 65+ working over 40 hours:', len(old_and_working))

target = df[df['prov'].isin(['Luanda', 'Benguela'])]
print('Rows in Luanda or Benguela:', len(target))

**Questions:**

- How many people report more than 100 usual hours? Are any sentinels left?
- Why must each condition be wrapped in parentheses?

**PT:** Quantas pessoas declaram mais de 100 horas? Restam sentinelas? Porque
cada condicao precisa de parenteses?

---

## Task 10: Detect missing values

Count them, express them as a share, and look at the shape before deciding.

**PT:** Conte, converta em percentagem, e observe o padrao antes de decidir.

In [ ]:
missing = pd.DataFrame({
    'n_missing':   # your code here
    'pct_missing':   # your code here, rounded to 1 decimal
})
missing.sort_values('pct_missing', ascending=False)

In [ ]:
counts = df.isna().sum()
counts[counts > 0].sort_values().plot(kind='barh', color='coral', figsize=(9, 6))
plt.title('Missing values by column')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

**Questions:**

- Which columns are most missing? Is that damage, or something else?
- What decides whether a gap is a problem?

**PT:** Que colunas tem mais valores em falta? E dano ou outra coisa? O que
decide se uma falha e um problema?

---

## Task 11: Missing by design is not missing by error

The textbook first move is `dropna()`. On a survey with skip patterns it is a
catastrophe. Measure it before you trust it.

**PT:** O primeiro reflexo e `dropna()`. Num inquerito com saltos de
questionario e uma catastrofe. Meca antes de confiar.

In [ ]:
print('Rows now:                   ', len(df))
print('Rows if we called dropna(): ', len(df.dropna()))

In [ ]:
# Only the identifiers are non negotiable
print('Before:', df.shape)
df = df.dropna(  # your code here: subset of identifier columns )
print('After: ', df.shape)

**Questions:**

- How many rows survive `dropna()`? Were you expecting that?
- Which columns are genuinely non negotiable?
- Would filling the labour columns with a median be reasonable? Why not?

**PT:** Quantas linhas sobrevivem a `dropna()`? Que colunas sao mesmo
obrigatorias? Faria sentido preencher as colunas de trabalho com a mediana?

---

## Task 12: Duplicates on a compound key

No two rows are identical, so `duplicated()` alone finds nothing. The real key is
`nidf` plus `ppno`: one row per person per household.

**PT:** Nenhuma linha e identica a outra, por isso `duplicated()` sozinho nao
encontra nada. A chave real e `nidf` mais `ppno`.

In [ ]:
print('Exact duplicate rows:', df.duplicated().sum())

dup_mask = df.duplicated(  # your code here: compound key, keep=False )
print('Rows sharing a person key:', dup_mask.sum())
df[dup_mask].sort_values(['nidf', 'ppno'])[
    ['nidf', 'ppno', 'dem_age', 'dem_sex', 'dem_rel', 'wkt_ushrstot']]

In [ ]:
# Keep the most complete record in each group
df['missing_count'] = df.isna().sum(axis=1)

print('Before:', df.shape)
df = (
    df
    .sort_values(['nidf', 'ppno', 'missing_count'])
    .drop_duplicates(  # your code here: compound key, keep='first' )
    .drop(columns='missing_count')
)
print('After: ', df.shape)

**Questions:**

- How many exact duplicates? How many rows share a person key?
- Why does the compound key find what `duplicated()` alone cannot?
- Look at the ages within a group. Is this really one person recorded twice?

**PT:** Quantos duplicados exatos? Porque a chave composta encontra o que
`duplicated()` sozinho nao encontra? Olhe as idades: e mesmo a mesma pessoa?

---

## Task 13: A rule that `describe()` cannot catch

Every household should have exactly one head. No summary statistic tells you
whether that holds, because it is a relationship between rows.

**PT:** Cada agregado deve ter exatamente um chefe. Nenhuma estatistica resumo
verifica isso, porque e uma relacao entre linhas.

In [ ]:
heads = df[df['dem_rel'] == 'Chefe/Pessoa de referência']
heads_per_household = heads['nidf'].  # your code here

print('Households with two or more heads:', (heads_per_household > 1).sum())
print('Households with no head recorded: ',
      df['nidf'].nunique() - heads_per_household.index.nunique())

**Questions:**

- How many households have two heads, and how many none?
- Run this before the deduplication too. Does the number change? Why?

**PT:** Quantos agregados tem dois chefes e quantos nenhum? Corra isto antes da
desduplicacao: o numero muda? Porque?

---

## Task 14: Save the cleaned dataset

Raw data is read only. Write to `10_cleaned/` and reload to confirm the round
trip.

**PT:** Os dados brutos sao apenas de leitura. Grave em `10_cleaned/` e recarregue
para confirmar.

In [ ]:
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

df = df.reset_index(drop=True)
df.  # your code here: to_csv with index=False
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'nidf': 'string', 'ppno': 'string',
                                     'g_06_id_iea': 'string'})
print('Reloaded:', check.shape)
print(check[['nidf', 'prov', 'ghvedt', 'mjt_syr']].dtypes)

**Questions:**

- How many rows does the cleaned file have? Can you account for every row lost?
- Reload it. Which dtypes did not survive, and why?
- What happened to the `descriptions` dictionary?

**PT:** Quantas linhas tem o ficheiro limpo? Consegue justificar cada linha
perdida? Que tipos nao sobreviveram? O que aconteceu ao dicionario
`descriptions`?